In [ ]:
from pathlib import Path

try:
    BASE_DIR = Path(__file__).resolve().parent
except NameError:
    BASE_DIR = Path.cwd()

# Agricultural Fields to RDF Knowledge Graph

This notebook demonstrates how agricultural polygon data (GeoJSON) can be transformed into a GeoSPARQL-compliant RDF knowledge graph.

The workflow includes loading geospatial data with GeoPandas, transforming coordinates to WGS84 (EPSG:4326), creating semantic entities with rdflib, and exporting the graph as Turtle.

In [ ]:
import geopandas as gpd
from rdflib import Graph, Namespace, Literal
from rdflib.namespace import RDF, RDFS, SKOS, XSD
from rdflib.namespace import NamespaceManager
import pandas as pd

In [ ]:
# import data / file paths

BASE_DIR = Path.cwd().parent  # eine Ebene nach oben

OUTPUT_DIR = BASE_DIR / "data" / "processed"

INPUT_FILE = BASE_DIR / "data" / "processed" / "examples" / "agrar_test.geojson"

OUTPUT_FILE = BASE_DIR / "data" / "processed" / "agro_individuals_with_geom.ttl"

In [ ]:
# import data / load GeoJSON

gdf = gpd.read_file(INPUT_FILE)

print("Features:", len(gdf))

print("CRS:", gdf.crs)



In [ ]:
# Set CRS to UTM for area calculation

gdf_utm = gdf.to_crs(25832)

# Add column "area" to gdf (in hectare /in Hektar)

gdf["area_ha"] = (gdf_utm.area / 10_000).round(2)



In [ ]:
# check the new GeoDataFrame

gdf.head()

In [ ]:



# Graph
g = Graph()

# Namespaces
AGRO = Namespace("http://www.semanticweb.org/frank/ontologies/2025/7/agro_geokg#")
GEO = Namespace("http://www.opengis.net/ont/geosparql#")

g.bind("agro", AGRO)
g.bind("geo", GEO)
g.bind("skos", SKOS)
g.bind("xsd", XSD)
g.bind("rdfs", RDFS)

# Untersuchungsgebiet
test_area_uri = AGRO["test_area_1"]

# Klasse AgriculturalArea beschreiben
g.add((AGRO.AgriculturalArea, RDF.type, RDFS.Class))

g.add((
    AGRO.AgriculturalArea,
    SKOS.prefLabel,
    Literal("Agricultural Area", lang="en")
))

g.add((
    AGRO.AgriculturalArea,
    SKOS.prefLabel,
    Literal("Landwirtschaftsfläche", lang="de")
))


# Polygone erzeugen
for idx, row in gdf.iterrows():

    agriculture_uri  = AGRO[f"agriculture_{idx+1}"]
    geom_uri = AGRO[f"geom_agriculture_{idx+1}"]

    # Klasse
    g.add((agriculture_uri, RDF.type, AGRO.AgriculturalArea))

    # Label
    g.add((
        agriculture_uri,
        SKOS.prefLabel,
        Literal(f"agriculture {idx+1}", lang="en")
    ))

    g.add((
        agriculture_uri,
        SKOS.prefLabel,
        Literal(f"Landwirtschaftsfläche {idx+1}", lang="de")
    ))

    # Geometrie-Klasse
    g.add((geom_uri, RDF.type, GEO.Geometry))

    # WKT-Geometrie
    g.add((
        geom_uri,
        GEO.asWKT,
        Literal(
            row.geometry.wkt,
            datatype=GEO.wktLiteral
        )
    ))

    # Verbindung Way zu Geometrie
    g.add((agriculture_uri, GEO.hasGeometry, geom_uri))

    # Beziehung zum Untersuchungsgebiet
    g.add((agriculture_uri, GEO.sfWithin, test_area_uri))

    
    
    # Area
    
    g.add((
    agriculture_uri,
    AGRO.area_ha,
    Literal(float(row['area_ha']), datatype=XSD.double)

    ))

    # Polygone-ID

    g.add((
    agriculture_uri,
    AGRO.featureID,
    Literal(int(idx))

    ))

    # Quelle hinzufügen
    
    ATKIS = AGRO.ATKIS

    g.add((ATKIS, SKOS.prefLabel, Literal("ATKIS", lang="en")))

    g.add((
    agriculture_uri,
    AGRO.dataSource,
    ATKIS
    ))


In [ ]:
g.serialize(destination=OUTPUT_FILE, format="turtle")

print("Features:", len(gdf))
print("Triples:", len(g))
print("Saved:", OUTPUT_FILE)